In [ ]:
!pip -q install flask

In [ ]:
%mkdir templates -p

In [ ]:
%%file templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>BEiT Image Classification</title>
</head>
<body>
    <h1>BEiT Image Classification</h1>
    <form action="/inference" method="post" enctype="multipart/form-data">
        <label for="image">Choose an image:</label><br>
        <input type="file" id="image" name="image" accept="image/*"><br><br>
        <button type="submit">Upload</button>
    </form>
    {% if prediction is defined %}
    <h2>Predicted Class: {{ prediction }}</h2>
    {% endif %}
</body>
</html>

In [ ]:
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(5005)"))

In [ ]:
%%file app.py
from flask import Flask, render_template, request
from transformers import AutoImageProcessor, AutoModelForImageClassification, pipeline
from PIL import Image
import torch

# Load pre-trained BEiT model and processor
processor = AutoImageProcessor.from_pretrained("microsoft/beit-base-patch16-224-pt22k-ft22k")
model = AutoModelForImageClassification.from_pretrained("microsoft/beit-base-patch16-224-pt22k-ft22k")

app = Flask(__name__)

# Default route to render the homepage
@app.route('/')
def home():
    """
    Renders the homepage “index.html” under the “templates” folder
    """
    return render_template('index.html')
    pass

# Route for model inference
@app.route('/inference', methods=['POST'])
def inference():
    """
    Model inference for the above pre-trained model.

    The output should be displayed in the same homepage as a text field.
    """
    # Get image file from HTML form
    image_file = request.files['image']
    image = Image.open(image_file)
    image = processor(image=image, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**image)

    predicted_class = processor.classnames[outputs.logits.argmax(-1).item()]
    predicted_probability = outputs.logits.softmax(-1).max().item()
    print(predicted_class)

    return render_template('index.html', prediction=predicted_class, probability=predicted_probability)
    pass

if __name__ == '__main__':
    app.run(host='127.0.0.1', port=5005)

In [ ]:
!python -m app

2024-05-15 08:56:49.601377: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-05-15 08:56:49.601428: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-05-15 08:56:49.602538: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-05-15 08:56:50.718004: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
preprocessor_config.json: 100% 276/276 [00:00<00:00, 1.69MB/s]
Could not find image processor class in the image processor config or the model config. Loading based on pattern matching with the model's feature extractor configuration. Please